# Static Test Prediction Collection

This notebook reproduces the published static test-set results and exports aligned per-sample predictions for the image, FEA, and multimodal models.

The current EmoHeVRDB subset structure is used for data loading. The legacy static models use a different class-ID order, so their outputs are remapped to the current canonical EmoHeVRDB order before evaluation and export.

The final table contains one row per image view (`sample_id`) and one shared identifier per underlying reenactment (`reenactment_id`). Each reenactment therefore occurs twice: once for the central view and once for the side view.


## 1. Setup

### 1.1 Imports and paths


In [1]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

DATASET_ROOT = Path('/workspace/datasets/emohevrdb')
MODEL_ROOT = Path('/workspace/emohevrdb-sfer/models')

SI_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-si' / 'test_set'
SFEA_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-sfea-as-csv' / 'test_set.csv'

IMAGE_MODEL_PATH = MODEL_ROOT / 'image_model.keras'
FEA_MODEL_PATH = MODEL_ROOT / 'fea_model.keras'
MULTIMODAL_MODEL_PATH = MODEL_ROOT / 'multimodal_model.keras'

IMAGE_SIZE = (224, 224)


2026-09-15 11:17:29.706555: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-15 11:17:29.706593: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-15 11:17:29.707240: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### 1.2 Class mappings

The legacy static models use the class order `Neutral, Happiness, Sadness, Surprise, Fear, Disgust, Anger`, whereas the current EmoHeVRDB subsets use `Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise`.


In [2]:
CANONICAL_ID_TO_EMOTION = {
    0: 'Anger',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happiness',
    4: 'Neutral',
    5: 'Sadness',
    6: 'Surprise',
}

CANONICAL_EMOTION_TO_ID = {v: k for k, v in CANONICAL_ID_TO_EMOTION.items()}

OLD_MODEL_ID_TO_EMOTION = {
    0: 'Neutral',
    1: 'Happiness',
    2: 'Sadness',
    3: 'Surprise',
    4: 'Fear',
    5: 'Disgust',
    6: 'Anger',
}

# For converting argmax(old-model-output) -> current EmoHeVRDB ID
OLD_TO_CANONICAL = np.array([4, 3, 5, 6, 2, 1, 0])

# For reordering complete probability vectors into current EmoHeVRDB order:
# [Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise]
CANONICAL_TO_OLD = np.array([6, 5, 4, 1, 0, 2, 3])


## 2. Load and align the static test data

### 2.1 Load the FEA test set

The source CSV uses `file_id` for the underlying reenactment identifier. It is renamed to `reenactment_id` immediately so that the identifier has the same semantic meaning throughout the analysis.


In [3]:
# Source file IDs have the following structure:
# <timestamp>-<set-id>-<participant-id>-<level-id>-<emoji-id>-<emotion-id>

fea_df = pd.read_csv(SFEA_TEST_PATH).rename(columns={'file_id': 'reenactment_id'})

print(f'fea_df shape: {fea_df.shape}')
display(fea_df.head())
print(f'fea_df columns: {fea_df.columns.tolist()}')

FEA_COLUMNS = [c for c in fea_df.columns if c not in {'reenactment_id', 'timestamp', 'Label'}]

assert len(fea_df) == 378
assert len(FEA_COLUMNS) == 63
assert fea_df['reenactment_id'].is_unique


fea_df shape: (378, 66)


,reenactment_id,timestamp,BrowLowererL,BrowLowererR,CheekPuffL,CheekPuffR,CheekRaiserL,CheekRaiserR,CheekSuckL,CheekSuckR,...,MouthRight,NoseWrinklerL,NoseWrinklerR,OuterBrowRaiserL,OuterBrowRaiserR,UpperLidRaiserL,UpperLidRaiserR,UpperLipRaiserL,UpperLipRaiserR,Label
0,1700478995850-2-1-1-0-0,1700478995850,4.174096e-02,0.043613,3.832706e-03,0.004610,0.043460,0.016550,1.401298e-45,1.401298e-45,...,2.395155e-12,2.410580e-03,2.882660e-03,1.083685e-15,0.000062,1.401298e-45,1.401298e-45,4.814688e-03,2.418802e-03,0
1,1700479004312-2-1-1-3-0,1700479004312,2.447398e-01,0.208823,5.883114e-03,0.005875,0.058813,0.033655,1.401298e-45,1.401298e-45,...,2.944601e-07,2.812969e-14,2.812969e-14,1.442060e-11,0.001512,1.401298e-45,1.401298e-45,6.511594e-03,8.534960e-03,0
2,1700479005401-2-1-1-4-0,1700479005401,3.127467e-01,0.281756,1.086564e-02,0.010870,0.070083,0.038030,1.401298e-45,1.401298e-45,...,9.621919e-03,2.722222e-19,2.722222e-19,2.714441e-14,0.000358,1.401298e-45,1.401298e-45,1.212497e-02,1.086563e-02,0
3,1700479253441-2-1-3-8-0,1700479253441,2.411928e-01,0.233531,5.881371e-03,0.008432,0.057930,0.030251,1.401298e-45,2.802597e-45,...,3.613444e-03,1.602602e-18,4.552448e-13,2.706358e-03,0.002941,1.401298e-45,1.401298e-45,3.094644e-04,1.164084e-02,0
4,1700479258342-2-1-3-10-0,1700479258342,5.683494e-07,0.000004,4.030551e-11,0.002698,0.044524,0.017329,1.425524e-13,1.084664e-13,...,2.497784e-07,5.280750e-14,3.639920e-14,2.224430e-02,0.020113,1.401298e-45,1.401298e-45,6.994161e-11,4.705341e-09,0


fea_df columns: ['reenactment_id', 'timestamp', 'BrowLowererL', 'BrowLowererR', 'CheekPuffL', 'CheekPuffR', 'CheekRaiserL', 'CheekRaiserR', 'CheekSuckL', 'CheekSuckR', 'ChinRaiserB', 'ChinRaiserT', 'DimplerL', 'DimplerR', 'EyesClosedL', 'EyesClosedR', 'EyesLookDownL', 'EyesLookDownR', 'EyesLookLeftL', 'EyesLookLeftR', 'EyesLookRightL', 'EyesLookRightR', 'EyesLookUpL', 'EyesLookUpR', 'InnerBrowRaiserL', 'InnerBrowRaiserR', 'JawDrop', 'JawSidewaysLeft', 'JawSidewaysRight', 'JawThrust', 'LidTightenerL', 'LidTightenerR', 'LipCornerDepressorL', 'LipCornerDepressorR', 'LipCornerPullerL', 'LipCornerPullerR', 'LipFunnelerLB', 'LipFunnelerLT', 'LipFunnelerRB', 'LipFunnelerRT', 'LipPressorL', 'LipPressorR', 'LipPuckerL', 'LipPuckerR', 'LipStretcherL', 'LipStretcherR', 'LipSuckLB', 'LipSuckLT', 'LipSuckRB', 'LipSuckRT', 'LipTightenerL', 'LipTightenerR', 'LipsToward', 'LowerLipDepressorL', 'LowerLipDepressorR', 'MouthLeft', 'MouthRight', 'NoseWrinklerL', 'NoseWrinklerR', 'OuterBrowRaiserL', 'Outer

### 2.2 Parse the static image test set

Each image filename additionally contains the camera index.

- `reenactment_id` identifies the underlying reenactment and is shared by both views.
- `sample_id` identifies one concrete image-view sample and is unique across the 756-row test set.


In [4]:
def parse_image_path(path: Path) -> dict:
    parts = path.stem.split('-')

    if len(parts) != 7:
        raise ValueError(f'Unexpected image filename: {path.name}')

    timestamp, set_id, participant_id, level_id, emoji_id, emotion_id, camera_index = parts

    reenactment_id = '-'.join(parts[:-1])

    return {
        'sample_id': path.stem,
        'reenactment_id': reenactment_id,
        'timestamp': int(timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'true_label_id': int(emotion_id),
        'camera_index': int(camera_index),
        'perspective': 'Central' if camera_index == '0' else 'Side',
        'image_path': str(path),
        'emotion_dir': path.parent.name,
    }


image_df = pd.DataFrame(parse_image_path(path) for path in SI_TEST_PATH.rglob('*.jpg'))

image_df = image_df.sort_values(['reenactment_id', 'camera_index']).reset_index(drop=True)


In [5]:
assert len(image_df) == 756
assert image_df['sample_id'].is_unique
assert image_df['reenactment_id'].nunique() == 378
assert set(image_df['camera_index']) == {0, 1}

assert (image_df.groupby('reenactment_id').size() == 2).all()
assert (image_df.groupby('reenactment_id')['camera_index'].nunique() == 2).all()
assert (image_df['emotion_dir'].map(CANONICAL_EMOTION_TO_ID) == image_df['true_label_id']).all()
assert set(image_df['reenactment_id']) == set(fea_df['reenactment_id'])


### 2.3 Verify labels and create the canonical prediction table

The FEA rows are joined to both corresponding image views using `reenactment_id`. The resulting `prediction_df` fixes the row order used for all subsequent inference.


In [6]:
fea_labels = fea_df.set_index('reenactment_id')['Label'].astype(int)

assert all(
    fea_labels.loc[row.reenactment_id] == row.true_label_id
    for row in image_df.itertuples()
)


In [7]:
prediction_df = image_df.merge(
    fea_df[['reenactment_id', *FEA_COLUMNS]],
    on='reenactment_id',
    how='left',
    validate='many_to_one'
)

prediction_df['true_label'] = prediction_df['true_label_id'].map(CANONICAL_ID_TO_EMOTION)

assert len(prediction_df) == 756
assert prediction_df['sample_id'].is_unique
assert prediction_df['reenactment_id'].nunique() == 378
assert prediction_df[FEA_COLUMNS].notna().all().all()


## 3. Inference utilities

### 3.1 Image preprocessing and deterministic datasets

No dataset is shuffled. The order of every inference dataset therefore remains aligned with `prediction_df`.


In [8]:
def parse_image(filename):
    image_string = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image_string, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image


In [9]:
def create_image_dataset(df, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(df['image_path'].values)
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [10]:
def create_fea_dataset(df, batch_size=32):
    X = df[FEA_COLUMNS].to_numpy(dtype=np.float32)
    return tf.data.Dataset.from_tensor_slices(X).batch(batch_size)


In [11]:
def create_multimodal_dataset(df, batch_size=32):
    image_paths = df['image_path'].values
    feas = df[FEA_COLUMNS].to_numpy(dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, feas))

    # The one-element outer tuple tells Keras that (image, fea) together form x
    # rather than interpreting the second tensor as y.
    ds = ds.map(
        lambda path, fea: ((parse_image(path), fea),),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


### 3.2 Prediction remapping and storage

All model outputs are converted from the legacy class order to the current canonical EmoHeVRDB class order before predictions and probabilities are stored.


In [12]:
def predictions_to_canonical(probabilities_old):
    probabilities_canonical = probabilities_old[:, CANONICAL_TO_OLD]
    predictions_canonical = np.argmax(probabilities_canonical, axis=1)

    return probabilities_canonical, predictions_canonical


In [13]:
def add_predictions(df, prefix, probabilities):
    probabilities, predictions = predictions_to_canonical(probabilities)

    df[f'{prefix}_pred_id'] = predictions
    df[f'{prefix}_pred'] = [CANONICAL_ID_TO_EMOTION[p] for p in predictions]

    emotions = [
        'Anger',
        'Disgust',
        'Fear',
        'Happiness',
        'Neutral',
        'Sadness',
        'Surprise',
    ]

    for i, emotion in enumerate(emotions):
        df[f'{prefix}_prob_{emotion.lower()}'] = probabilities[:, i]

    return predictions


## 4. Static image model

### 4.1 Load the frozen model and run inference


In [14]:
image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH, compile=False)

print('Input:', image_model.input_shape)
print('Output:', image_model.output_shape)

image_probabilities_old = image_model.predict(
    create_image_dataset(prediction_df, batch_size=32),
    verbose=1
)

image_predictions = add_predictions(prediction_df, 'image', image_probabilities_old)


Input: (None, 224, 224, 3)
Output: (None, 7)
24/24 [==============================] - 2s 21ms/step


### 4.2 Verify the published image result

Expected result: **528 / 756 = 69.84%**.


In [15]:
image_correct = np.sum(image_predictions == prediction_df['true_label_id'].to_numpy())
image_accuracy = image_correct / len(prediction_df)

print(f'Image: {image_correct}/756 = {image_accuracy:.6f}')

assert image_correct == 528
assert (prediction_df.groupby('reenactment_id')['camera_index'].nunique().eq(2).all())


Image: 528/756 = 0.698413


In [16]:
del image_model
tf.keras.backend.clear_session()
gc.collect()


19290

## 5. Static FEA model

### 5.1 Load the frozen model and run inference

The same FEA vector is paired with the central- and side-view image rows of each reenactment. Therefore, the FEA prediction must be identical across both rows belonging to the same `reenactment_id`.


In [17]:
fea_model = tf.keras.models.load_model(FEA_MODEL_PATH, compile=False)

print('Input:', fea_model.input_shape)
print('Output:', fea_model.output_shape)

fea_probabilities_old = fea_model.predict(
    create_fea_dataset(prediction_df, batch_size=32),
    verbose=1
)

fea_predictions = add_predictions(prediction_df, 'fea', fea_probabilities_old)


Input: (None, 63)
Output: (None, 7)
24/24 [==============================] - 0s 1ms/step


### 5.2 Verify the published FEA result

Expected result: **271 / 378 = 71.69%**, represented here as **542 / 756** because each reenactment occurs once per image view.


In [18]:
fea_correct = np.sum(fea_predictions == prediction_df['true_label_id'].to_numpy())
fea_accuracy = fea_correct / len(prediction_df)

print(f'FEA paired representation: {fea_correct}/756 = {fea_accuracy:.6f}')

assert fea_correct == 542
assert (prediction_df.groupby('reenactment_id')['fea_pred_id'].nunique().eq(1).all())


FEA paired representation: 542/756 = 0.716931


In [19]:
del fea_model
tf.keras.backend.clear_session()
gc.collect()


1102

## 6. Static multimodal model

### 6.1 Define the custom cross-attention layer


In [20]:
from tensorflow.keras.layers import Layer, Dense
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.saving import register_keras_serializable


@register_keras_serializable()
class CrossAttention(Layer):

    def __init__(self, units_v1, units_v2, **kwargs):
        super().__init__(**kwargs)
        self.units_v1 = units_v1
        self.units_v2 = units_v2

        self.weight_layer_v1 = Dense(
            units_v1,
            activation='sigmoid',
            kernel_initializer=GlorotUniform()
        )

        self.weight_layer_v2 = Dense(
            units_v2,
            activation='sigmoid',
            kernel_initializer=GlorotUniform()
        )

    def call(self, inputs, training=False):
        v1, v2 = inputs

        weights_v1 = self.weight_layer_v1(v2)
        weights_v2 = self.weight_layer_v2(v1)

        weighted_v1 = weights_v1 * v1
        weighted_v2 = weights_v2 * v2

        return [weighted_v1, weighted_v2]


### 6.2 Load the frozen multimodal model


In [21]:
custom_objects = {
    'CrossAttention': CrossAttention,
    'Custom>CrossAttention': CrossAttention,
}

multimodal_model = tf.keras.models.load_model(
    MULTIMODAL_MODEL_PATH,
    custom_objects=custom_objects,
    compile=False,
    safe_mode=False
)

print('Inputs:', multimodal_model.input_shape)
print('Output:', multimodal_model.output_shape)

for inp in multimodal_model.inputs:
    print(inp.name, inp.shape)


Inputs: [(None, 224, 224, 3), (None, 63)]
Output: (None, 7)
input_1 (None, 224, 224, 3)
fau_input (None, 63)


### 6.3 Run inference and verify the published multimodal result

Expected result: **608 / 756 = 80.42%**.


In [22]:
multimodal_probabilities_old = multimodal_model.predict(
    create_multimodal_dataset(prediction_df, batch_size=32),
    verbose=1
)

multimodal_predictions = add_predictions(
    prediction_df,
    'multimodal',
    multimodal_probabilities_old
)

multimodal_correct = np.sum(
    multimodal_predictions == prediction_df['true_label_id'].to_numpy()
)
multimodal_accuracy = multimodal_correct / len(prediction_df)

print(f'Multimodal: {multimodal_correct}/756 = {multimodal_accuracy:.6f}')

assert multimodal_correct == 608


24/24 [==============================] - 1s 16ms/step
Multimodal: 608/756 = 0.804233


In [23]:
del multimodal_model
tf.keras.backend.clear_session()
gc.collect()


4644

## 7. Final validation and export

### 7.1 Integrity checks

These checks verify the expected number of view-level samples and reenactments, balanced class counts, complete predictions, and identical FEA predictions across the two views of each reenactment.


In [24]:
assert len(prediction_df) == 756
assert prediction_df['sample_id'].is_unique
assert prediction_df['reenactment_id'].nunique() == 378

assert (prediction_df.groupby('reenactment_id').size() == 2).all()
assert (prediction_df.groupby('reenactment_id')['camera_index'].nunique() == 2).all()
assert (prediction_df.groupby('true_label_id').size() == 108).all()

assert prediction_df[['image_pred_id', 'fea_pred_id', 'multimodal_pred_id']].notna().all().all()

assert (prediction_df.groupby('reenactment_id')['fea_pred_id'].nunique().eq(1).all())


### 7.2 Summarize reproduced results


In [25]:
for model in ['image', 'fea', 'multimodal']:
    correct = (prediction_df[f'{model}_pred_id'] == prediction_df['true_label_id']).sum()

    print(
        f'{model:<12}: '
        f'{correct:>3}/{len(prediction_df)} '
        f'= {correct / len(prediction_df):.4%}'
    )


image       : 528/756 = 69.8413%
fea         : 542/756 = 71.6931%
multimodal  : 608/756 = 80.4233%


### 7.3 Export per-sample predictions

The exported CSV keeps `sample_id` as the unique view-level key and `reenactment_id` as the shared key linking both views of the same reenactment. Raw FEA inputs are omitted; model predictions and class probabilities are retained.


In [26]:
output_columns = [
    'sample_id',
    'reenactment_id',
    'timestamp',
    'set_id',
    'participant_id',
    'level_id',
    'emoji_id',
    'camera_index',
    'perspective',
    'true_label_id',
    'true_label',
    'image_path',
]

prediction_columns = [
    c for c in prediction_df.columns
    if (
        c.startswith('image_')
        or c.startswith('fea_')
        or c.startswith('multimodal_')
    )
    and c != 'image_path'
]

output_df = prediction_df[output_columns + prediction_columns].copy()

output_df.to_csv('static_test_predictions.csv', index=False)

print(f'Exported {len(output_df)} rows to static_test_predictions.csv')


Exported 756 rows to static_test_predictions.csv
